In [1]:
import mlflow
import os

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("agent10_tone_param_adjust")

2025/12/24 11:04:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/24 11:04:56 INFO mlflow.store.db.utils: Updating database tables
2025/12/24 11:04:56 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/24 11:04:56 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/24 11:04:56 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/24 11:04:56 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='/Users/mac/Desktop/project/STUDY-DATA/third_week/12_24/mlruns/1', creation_time=1766521322955, experiment_id='1', last_update_time=1766521322955, lifecycle_stage='active', name='agent10_tone_param_adjust', tags={}>

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score

import mlflow
import mlflow.sklearn

In [3]:
# tone centroid (이미 확정된 것)
tone_centroids = np.load("../data_csv/tone_centroids.npy")  # shape: (T, D)
tone_ids = pd.read_csv("../data_csv/tone_centroids_meta.csv")["tone_id"].tolist()

# rule-based 파라미터 테이블
tone_param_df = pd.read_csv("../data_csv/tone_param_profile.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../data_csv/tone_centroids.npy'

In [ ]:
rows = []

for i, tone_id in enumerate(tone_ids):
    vec = tone_centroids[i]
    params = tone_param_df[tone_param_df["tone_id"] == tone_id].iloc[0]

    row = {
        "tone_id": tone_id,
        **{f"v_{j}": vec[j] for j in range(len(vec))},
        "proof_level": params["proof_level"],
        "emotion_level": params["emotion_level"],
        "cta_strength": params["cta_strength"],
        "sentence_len": params["sentence_len"]
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.head()

In [ ]:
X = df.filter(regex="^v_").values

y_proof   = df["proof_level"].values
y_emotion = df["emotion_level"].values
y_cta     = df["cta_strength"].values

In [ ]:
mlflow.set_experiment("agent10_tone_param_adjust")

with mlflow.start_run(run_name="ridge_regression"):

    model = Ridge(alpha=1.0)
    model.fit(X, y_proof)

    preds = model.predict(X)
    rmse = mean_squared_error(y_proof, preds, squared=False)

    mlflow.log_param("model", "Ridge")
    mlflow.log_metric("rmse_proof", rmse)
    mlflow.sklearn.log_model(model, "ridge_proof")

    print("RMSE (proof_level):", rmse)

In [ ]:
with mlflow.start_run(run_name="logistic_emotion"):

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, y_emotion)

    preds = clf.predict(X)
    acc = accuracy_score(y_emotion, preds)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_metric("acc_emotion", acc)
    mlflow.sklearn.log_model(clf, "logistic_emotion")

    print("Accuracy (emotion_level):", acc)

In [ ]:
print("""
해석 가이드:

1) RMSE / Accuracy가 의미 있게 나오면
   → tone_vector → 파라미터 보정 가능

2) 성능이 낮으면
   → rule-based 고정 유지 (ML 불필요)

3) 혼합 전략 가능:
   rule_value + α * ml_adjustment
""")